# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'dataset.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

df = pl.read_parquet(DATA_PATH)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Ukuran awal: 2,475,926 baris x 5 kolom


halaman_asal,halaman_tujuan,tipe_link,jumlah_klik,periode
str,str,str,i64,str
"""other-empty""","""Halaman_Utama""","""external""",683510,"""2024-08"""
"""other-empty""","""Kleopatra""","""external""",238768,"""2024-08"""
"""other-internal""","""Partai_Kasih""","""external""",195370,"""2024-08"""
"""other-search""","""Bahlil_Lahadalia""","""external""",155770,"""2024-08"""
"""other-empty""","""DuckDuckGo""","""external""",138430,"""2024-08"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""halaman_asal""","""VARCHAR""","""""Gadis_\""Hawk_Tuah\""""""","""＃2019GantiPresiden""",59507,null,null,null,null,null,2475926,0.00
"""halaman_tujuan""","""VARCHAR""","""!Kung""","""＃LoveSignal""",338191,null,null,null,null,null,2475926,0.00
"""tipe_link""","""VARCHAR""","""external""","""other""",2,null,null,null,null,null,2475926,0.00
"""jumlah_klik""","""BIGINT""","""10""","""3055641""",7631,"""97.57704753696193""","""2428.5885475029536""","""14""","""22""","""51""",2475926,0.00
"""periode""","""VARCHAR""","""2024-08""","""2026-08""",4,null,null,null,null,null,2475926,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
if len(null_summary) == 0:
    print('Keterangan: Tidak ditemukan missing values (seluruh kolom 100% terisi lengkap / 0 null).')
null_summary

Keterangan: Tidak ditemukan missing values (seluruh kolom 100% terisi lengkap / 0 null).


kolom,jumlah_null,persen_null
str,u32,f64


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [4]:
# Menangani potensi missing values serta standardisasi format teks (mengganti '_' menjadi spasi)
df_missing_handled = df.with_columns([
    # Kolom kategorik: siapkan default value jika null, serta rapikan format spasi teks
    pl.col('halaman_asal').fill_null('other-empty').str.replace_all('_', ' '),
    pl.col('halaman_tujuan').fill_null('tidak diketahui').str.replace_all('_', ' '),
    pl.col('tipe_link').fill_null('other'),
    pl.col('periode').fill_null('unknown'),
    # Kolom numerik: siapkan fallback median jika ada null
    pl.col('jumlah_klik').fill_null(pl.col('jumlah_klik').median()),
])

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')
print('Keterangan: Dataset mentah memang 0 null; fill_null() diterapkan sebagai defensive cleaning.')

Null sebelum: 0
Null sesudah: 0
Keterangan: Dataset mentah memang 0 null; fill_null() diterapkan sebagai defensive cleaning.


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [5]:
# Kolom numerik utama pada dataset Wikimedia Clickstream
KOLOM_NUMERIK = 'jumlah_klik'

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
# Batas bawah domain knowledge: jumlah klik minimal bernilai 1
batas_bawah = max(1.0, float(q1 - 1.5 * iqr))
batas_atas = float(q3 + 1.5 * iqr)

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')

# Winsorization: clip nilai ke batas yang valid, bukan menghapus barisnya
df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=int(batas_bawah), upper_bound=int(batas_atas))
)

Q1=14.0, Q3=50.0, IQR=36.0
Batas bawah: 1.0, Batas atas: 104.0


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [6]:
# Kolom kunci bisnis: relasi unik antara halaman asal, tujuan, tipe tautan, dan periode observasi bulanan
KOLOM_KUNCI = ['halaman_asal', 'halaman_tujuan', 'tipe_link', 'periode']

total_baris = len(df_outlier_handled)
baris_unik = (
    df_outlier_handled.select(KOLOM_KUNCI).n_unique()
    if isinstance(KOLOM_KUNCI, list)
    else (
        df_outlier_handled[KOLOM_KUNCI].n_unique()
        if KOLOM_KUNCI in df_outlier_handled.columns
        else total_baris
    )
)
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan kombinasi {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=KOLOM_KUNCI if isinstance(KOLOM_KUNCI, list) else ([KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None),
    keep='first',
    maintain_order=True,
)

Total baris: 2,475,926
Baris unik berdasarkan kombinasi ['halaman_asal', 'halaman_tujuan', 'tipe_link', 'periode']: 2,475,926
Duplikat: 0


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [7]:
connection.register('dataset_bersih', df_dedup)

# Aturan validasi: integritas null, rentang batas numerik, dan aturan domain klik > 0
validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK} < {batas_bawah} OR {KOLOM_NUMERIK} > {batas_atas}
            THEN 1 ELSE 0 END) AS pelanggaran_batas,
        SUM(CASE WHEN {KOLOM_NUMERIK} <= 0 THEN 1 ELSE 0 END) AS pelanggaran_klik_negatif
    FROM dataset_bersih
""").pl()
validasi

total_baris,pelanggaran_null,pelanggaran_batas,pelanggaran_klik_negatif
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]"
2475926,0,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 2,475,926
Baris sesudah cleaning: 2,475,926
Data bersih tersimpan di: C:\UMM\Semester 7\Analisis Big Data\tugas1-mnabilfs\data\processed\dataset_bersih.parquet


## Catatan Milestone 2

### 1. Dimensi Kualitas Data yang Ditangani
- **Kelengkapan (Completeness)**: Dipastikan tidak ada nilai `null` pada kolom numerik maupun kategorik. Fallback disiapkan secara defensif (`'other-empty'`, `'tidak diketahui'`, `'other'`, dan median).
- **Konsistensi Sintaksis (Syntactic Consistency)**: Karakter garis bawah (`_`) pada nama artikel `halaman_asal` dan `halaman_tujuan` distandarisasi menjadi spasi agar representatif dan siap digunakan pada visualisasi graf / teks di Milestone 3.
- **Keunikan & Integritas Kunci (Uniqueness)**: Hubungan observasi diidentifikasi menggunakan kunci bisnis komposit `['halaman_asal', 'halaman_tujuan', 'tipe_link', 'periode']`, menghasilkan **2.475.926 baris transaksi unik** yang mempertahankan runtun waktu untuk Milestone 3.
- **Validitas Nilai & Outlier (Validity)**: Kolom `jumlah_klik` memiliki sebaran sangat skewed dengan nilai maksimum mencapai jutaan klik. Diterapkan teknik **Winsorization (clipping)** dengan batas IQR `[1.0, 104.0]`. Nilai ekstrem diclip ke batas valid sehingga mempertahankan ukuran sampel data analitik tanpa membuang baris data.

### 2. Perubahan Data
- **Jumlah baris awal (sebelum cleaning)**: 2.475.926 baris.
- **Jumlah baris akhir (sesudah cleaning)**: **2.475.926 baris** (seluruh observasi bulanan terjaga utuh, memenuhi kriteria > 1 juta baris).

### 3. Hasil Validasi Integritas Data (DuckDB)
- **Pelanggaran Null**: 0 (100% data terisi).
- **Pelanggaran Batas Outlier**: 0 (seluruh baris berada dalam rentang valid [1, 104]).
- **Pelanggaran Klik Non-Positif**: 0 (semua klik bernilai valid positif).
- **Penyimpanan**: File disimpan di `data/processed/dataset_bersih.parquet` (format Parquet kolumnar, siap untuk query dan visualisasi cepat pada Milestone 3).